# Leaky RNN vs. Vanilla RNN on the Vanishing/Exploding Gradient Benchmarks

This notebook re-runs the benchmarks from `gradient_fixes_demo.ipynb`, but
instead of adding Pascanu, Mikolov & Bengio's (2013) training-time fixes
(gradient clipping + norm-preserving regularizer) to the vanilla RNN, it
changes the **model**: a standard **leaky RNN**, the Euler-discretized
rate network common in computational neuroscience (e.g. Song, Yang & Wang,
2016):

$$h_t = (1-\alpha)\,h_{t-1} + \alpha\,\tanh(W_{xh}x_t + W_{hh}h_{t-1} + b), \qquad \alpha = \Delta t/\tau$$

With $\alpha = 1$ this is exactly the vanilla RNN in `rnn.py`. With
$\alpha < 1$ each unit is a leaky integrator with time constant
$\tau = \Delta t/\alpha$ steps. Here $\alpha = 0.1$ (τ ≈ 10 steps). The
one-step Jacobian becomes

$$\frac{\partial h_t}{\partial h_{t-1}} = (1-\alpha)\,I + \alpha\,\mathrm{diag}(1-a_t^2)\,W_{hh},$$

an identity "leak" path plus a shrunken copy of the vanilla Jacobian.
That's the only change. No clipping and no regularizer are used anywhere
in this notebook. The model is in `leaky_rnn.py`; the tasks, seeds,
optimizers and hyperparameters are the same as `gradient_fixes_demo.ipynb`, so
the vanilla numbers below match that notebook's baseline exactly.

As in the fixes notebook, the results are reported as they came out: the
leak helps a lot in one place, does nothing in another, and makes one task
**worse**.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from biornn.models.rnn import VanillaRNNCell, AddingRNN, CopyRNN, set_spectral_radius
from biornn.models.leaky_rnn import LeakyRNNCell, LeakyAddingRNN, LeakyCopyRNN
from biornn.tasks import adding_problem, copy_task
from biornn.device import get_device
from biornn.visualize import plot_gradient_norms, plot_comparison_bars

ALPHA = 0.1  # dt / tau: each unit integrates over ~10 timesteps

# Training loops run on DEVICE. `BIORNN_DEVICE=cpu` overrides it; see biornn/device.py
# for why the CPU can be faster for these small models.
DEVICE = get_device()
print('training on', DEVICE)

## 1. Why a leak should help: backpropagated gradient norm at initialization

Before training anything, roll each cell forward 60 steps on random input
and measure $\|\partial h_T/\partial h_t\|$ as a function of lag, as in
the original demo. We use two recurrent-weight scalings: spectral radius
0.5 (the vanilla vanishing regime) and 3.0 (the vanilla exploding regime
used in Section 2).


In [ ]:
def jacobian_norms(cell, T=60, seed=0):
    """||dh_T / dh_t|| for t = T..0, from a single random-input rollout."""
    torch.manual_seed(seed)
    h = torch.zeros(1, cell.hidden_size, requires_grad=True)
    hidden_states = [h]
    x = torch.randn(T, 1, 1)
    for t in range(T):
        h = cell(x[t], hidden_states[-1])
        h.retain_grad()
        hidden_states.append(h)
    hidden_states[-1].pow(2).sum().backward()
    return [hs.grad.norm().item() for hs in hidden_states][::-1]  # index by lag T-t

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, radius in zip(axes, [0.5, 3.0]):
    for label, make_cell, color in [
        ('vanilla (alpha=1)', lambda: VanillaRNNCell(1, 20), '#c1121f'),
        (f'leaky (alpha={ALPHA})', lambda: LeakyRNNCell(1, 20, alpha=ALPHA), '#219ebc'),
    ]:
        torch.manual_seed(0)
        cell = make_cell()
        set_spectral_radius(cell, radius)
        norms = jacobian_norms(cell)
        ax.plot(norms, marker='o', markersize=3, label=label, color=color)
        print(f'radius={radius}  {label:<18}  ||dh_T/dh_t|| at lag 10: {norms[10]:.2e}   lag 50: {norms[50]:.2e}')
    ax.set_yscale('log')
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel('time lag  (T - t)')
    ax.set_title(f'W_hh spectral radius = {radius}')
    ax.legend()
axes[0].set_ylabel(r'$\|\partial h_T / \partial h_t\|$')
fig.tight_layout()
plt.show()

**Radius 0.5:** the vanilla gradient vanishes geometrically. It is about
$3\times10^{-5}$ by lag 10 and $5\times10^{-23}$ by lag 50, and it underflows to exactly zero
after that. No learning signal effectively survives past a few steps. The leaky cell's gradient
stays of order 1 at lag 10 and is still about $6\times10^{-2}$ at lag
50. That is roughly 20 orders of magnitude more. The $(1-\alpha)I$ term
gives the backward pass a path that decays only like $(1-\alpha)^k$,
spread over τ ≈ 10 steps, instead of the vanilla cell's
$(\text{radius}\cdot\tanh')^k$.

**Radius 3.0:** the leak also pulls the Jacobian toward identity from
above. Its effective gain is about $(1-\alpha) + \alpha\cdot 3$ per step,
not 3, and `tanh` saturation brings it down further. The leaky gradient
therefore stays within a small factor of 1 across all 60 lags. The vanilla
curve climbs steadily with lag, reaching about 200× at lag 50 and
about $10^3$ by lag 60.


## 2. Exploding gradients: spectral radius 3.0, plain SGD, no clipping

Same setup as Section 1 of `gradient_fixes_demo.ipynb`: recurrent weights
initialized at spectral radius 3.0, plain SGD at `lr=1e-2`, 15 epochs,
same seed and data stream. The only difference between the two runs is
$\alpha$. **Neither run uses clipping.**


In [ ]:
def run_exploding_demo(alpha, seed=1, epochs=15, lr=1e-2):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = AddingRNN(hidden_size=32).to(DEVICE) if alpha == 1.0 else LeakyAddingRNN(hidden_size=32, alpha=alpha).to(DEVICE)
    set_spectral_radius(model.cell, 3.0)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    raw_norms, losses = [], []
    for epoch in range(epochs):
        x, y = adding_problem(64, 100, rng=rng, device=DEVICE)
        opt.zero_grad()
        pred = model(x)
        loss = nn.functional.mse_loss(pred, y)
        loss.backward()  # no clipping for either model
        raw_norms.append(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf')).item())
        opt.step()
        losses.append(loss.item())
    return raw_norms, losses

norm_vanilla, loss_vanilla = run_exploding_demo(alpha=1.0)
norm_leaky, loss_leaky = run_exploding_demo(alpha=ALPHA)

print('vanilla -- raw grad norm per epoch:', [f'{v:.2e}' for v in norm_vanilla])
print('leaky   -- raw grad norm per epoch:', [f'{v:.2e}' for v in norm_leaky])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(norm_vanilla, marker='o', label='vanilla (alpha=1)', color='#c1121f')
ax1.plot(norm_leaky, marker='o', label=f'leaky (alpha={ALPHA})', color='#219ebc')
ax1.set_yscale('log')
ax1.set_xlabel('epoch')
ax1.set_ylabel('raw gradient norm (log scale)')
ax1.set_title('Spectral radius 3.0, no clipping: raw gradient norm')
ax1.legend()

ax2.plot(loss_vanilla, marker='o', label='vanilla (alpha=1)', color='#c1121f')
ax2.plot(loss_leaky, marker='o', label=f'leaky (alpha={ALPHA})', color='#219ebc')
ax2.set_xlabel('epoch')
ax2.set_ylabel('loss')
ax2.set_title('Loss over the same 15 epochs')
ax2.legend()

fig.tight_layout()
plt.show()

The vanilla RNN reproduces the explosion from the original notebook: a
first-step gradient norm of about $3.7\times10^{11}$. The leaky RNN starts
from the same $W_{hh}$ and gets about **12** on its first step, roughly
10 orders of magnitude smaller, with no clipping. It then decays smoothly
to O(0.1–1). In the fixes notebook, clipping only bounded the *applied
update* while the raw gradient stayed around $10^{11}$ every epoch. Here
the raw gradient itself is well-behaved, because the leak keeps the
dynamics from being chaotic in the first place.

In the loss panel, neither model learns much in 15 SGD epochs. Both settle
around the predict-the-mean level (~0.17). The leaky model gets there
without the single huge, saturation-driven jump that the vanilla run
depends on. That jump is the "lucky" step discussed in the fixes notebook.


## 3. Vanishing gradients: the Adding Problem

Same sweep and budget as `gradient_fixes_demo.ipynb`: `seq_len` ∈ {10, 50,
150}, Adam at `lr=1e-2`, 2000 epochs, hidden size 32, seed 42. Each bar
pair differs only in $\alpha$.


In [ ]:
def train_adding(seq_len, alpha, seed=42, epochs=2000, hidden_size=32, lr=1e-2):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    if alpha == 1.0:
        model = AddingRNN(hidden_size=hidden_size).to(DEVICE)
    else:
        model = LeakyAddingRNN(hidden_size=hidden_size, alpha=alpha).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        x, y = adding_problem(128, seq_len, rng=rng, device=DEVICE)
        opt.zero_grad()
        pred = model(x)
        loss = nn.functional.mse_loss(pred, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

seq_lens = [10, 50, 150]
vanilla_final, leaky_final = [], []
for seq_len in seq_lens:
    vanilla_losses = train_adding(seq_len, alpha=1.0)
    leaky_losses = train_adding(seq_len, alpha=ALPHA)
    vanilla_final.append(vanilla_losses[-1])
    leaky_final.append(leaky_losses[-1])
    print(f'seq_len={seq_len:>4}  vanilla={vanilla_losses[-1]:.4f}  leaky={leaky_losses[-1]:.4f}')

fig = plot_comparison_bars(
    seq_lens, vanilla_final, leaky_final,
    ylabel='final MSE', title='Adding Problem: vanilla vs. leaky RNN',
    baseline_label='vanilla RNN (alpha=1)', fixed_label=f'leaky RNN (alpha={ALPHA})',
    reference=0.167, reference_label='predict-the-mean baseline',
)
plt.show()

- **seq_len=10:** both solve it. The leaky model is slightly better
  (0.0005 vs 0.0012). Unlike the regularizer, which made this case worse
  (0.0315), the leak adds no competing loss term, so there's no cost.
- **seq_len=50:** the biggest effect. The vanilla RNN sits near the
  predict-the-mean baseline (0.155), and the leaky RNN learns the task
  (**0.020**). For comparison, clipping + regularizer only got to 0.091 on
  this same setting. The first marker is ~25 steps from the end, a few
  leak time constants, which is within the range where Section 1 showed
  the leaky gradient survives.
- **seq_len=150:** both fail (0.195 vs 0.191). The first marker is now
  ~75+ steps back, about 7–8 τ, and $(0.9)^{75} \approx 4\times10^{-4}$.
  The leak extends the gradient horizon by a constant factor; it doesn't
  remove the horizon. (A separate check with α=0.05, τ≈20, also failed
  at 150 in this budget.)


## 4. Copy Task

Same settings as the fixes notebook: pattern length 1, 8 categories,
delays {0, 20, 60}, 400 Adam epochs. We also print which symbol each model
outputs at the recall step (index 0 = blank, 1–8 = categories, 9 =
delimiter) to see *how* it fails.


In [ ]:
def train_copy(delay, alpha, seed=42, epochs=400, hidden_size=32, lr=5e-3,
               pattern_len=1, n_categories=8):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    if alpha == 1.0:
        model = CopyRNN(hidden_size=hidden_size, n_categories=n_categories).to(DEVICE)
    else:
        model = LeakyCopyRNN(hidden_size=hidden_size, n_categories=n_categories, alpha=alpha).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        x, y = copy_task(64, pattern_len, delay, n_categories, rng=rng, device=DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        loss.backward()
        opt.step()
    x, y = copy_task(512, pattern_len, delay, n_categories, rng=rng, device=DEVICE)
    with torch.no_grad():
        logits = model(x)
    pred = logits[:, -pattern_len:, :].argmax(dim=-1)
    acc = (pred == y[:, -pattern_len:]).float().mean().item()
    # which symbols does the model emit at the recall step? (0 = blank)
    counts = torch.bincount(pred.flatten(), minlength=model.n_symbols).tolist()
    return acc, counts

delays = [0, 20, 60]
vanilla_acc, leaky_acc = [], []
for delay in delays:
    v_acc, v_counts = train_copy(delay, alpha=1.0)
    l_acc, l_counts = train_copy(delay, alpha=ALPHA)
    vanilla_acc.append(v_acc)
    leaky_acc.append(l_acc)
    print(f'delay={delay:>3}  vanilla acc={v_acc:.3f}  leaky acc={l_acc:.3f}')
    print(f'           recall-step predictions by symbol -- vanilla: {v_counts}  leaky: {l_counts}')

fig = plot_comparison_bars(
    delays, vanilla_acc, leaky_acc,
    ylabel='recall accuracy', title='Copy Task: vanilla vs. leaky RNN',
    baseline_label='vanilla RNN (alpha=1)', fixed_label=f'leaky RNN (alpha={ALPHA})',
    reference=1/8, reference_label='chance level',
)
plt.show()

**Here the leak makes things worse.** At `delay=0` both models copy
perfectly. At `delay=20` and `delay=60` the vanilla RNN falls back to a
constant guess, always emitting category 2 (0.154, about chance). The
leaky RNN gets **0.000**: it emits *blank* at the recall step for all 512
test sequences.

The prediction counts point to the cause. The task wants the output to
change from "blank" to "the remembered symbol" **in the one step right
after the delimiter**. With τ ≈ 10 steps, one step of input moves the
hidden state by only α = 10% of the way toward its new target, so the
readout can't flip within that single step. Reading "blank" everywhere is
the easy minimum of the per-timestep cross-entropy, and the model stays
there. With delay 0 the symbol and delimiter are next to each other, so
this doesn't come up. (Extending training to 1500 epochs, not shown, left
the leaky model still at 0.000.)

So the leak trades **temporal resolution for memory**. Integration that
keeps gradients alive over tens of steps also means the network can't
respond sharply within one step. Tasks that need both (fast readout, long
memory) are what gating was designed for: an LSTM/GRU learns a
*per-unit, input-dependent* α, so it can hold a value for a long time and
still switch in one step.


## 5. Takeaways

| | Vanilla | + clipping & regularizer | Leaky (α=0.1) |
|---|---|---|---|
| Raw grad norm, radius 3, epoch 0 | 3.7e11 | 3.7e11 (update capped) | **12** |
| Adding, seq_len=10 | 0.0012 | 0.0315 | **0.0005** |
| Adding, seq_len=50 | 0.155 | 0.091 | **0.020** |
| Adding, seq_len=150 | 0.195 | 0.194 | 0.191 |
| Copy, delay=20 | 0.154 | 0.154 | **0.000** |
| Copy, delay=60 | 0.154 | **0.244** | **0.000** |

*(Middle column from `gradient_fixes_demo.ipynb`, same seeds.)*

- **A leak is an architectural fix, not a training trick.** The $(1-\alpha)I$
  term puts the recurrent Jacobian near identity at initialization. That
  reduces both vanishing (Section 1, radius 0.5) and exploding (Sections 1–2,
  radius 3.0) without clipping or a regularizer.
- **On the Adding Problem it beats the Pascanu fixes clearly** in the
  moderate regime (seq_len=50: 0.020 vs 0.091), and it doesn't hurt the
  easy case.
- **It extends the gradient horizon by roughly τ; it doesn't remove it.**
  At seq_len=150 every variant fails.
- **It costs temporal resolution.** On the Copy Task, which needs a
  one-step response after the cue, the leaky net does worse than chance
  and worse than the vanilla net, because it can't change its output fast
  enough. A single fixed α can't give both long memory and fast responses.
  Learned, input-dependent gating (LSTM/GRU), or a spread of per-unit time
  constants, can.
